In [2]:
import requests
import time
import csv
from rdflib import Graph, Namespace, URIRef, OWL, RDFS, RDF, Literal

MV  = Namespace("http://moviekg.local/resource/")
MVO = Namespace("http://moviekg.local/ontology/")
WD  = Namespace("http://www.wikidata.org/entity/")

In [3]:
# Load previously generated KB
g = Graph()
g.parse("movie_kb.ttl", format="turtle")

# Create a new graph for alignment triples
align_g = Graph()
align_g.bind("mv", MV)
align_g.bind("wd", WD)
align_g.bind("owl", OWL)

mapping = []   # Store mapping table

# Query all Movies
movies = list(g.subjects(predicate=RDF.type, object=MVO.Movie))
persons = list(g.subjects(predicate=RDF.type, object=MVO.Person))
print(f"Movies: {len(movies)}, Persons: {len(persons)}")

Movies: 163, Persons: 873


In [4]:
def search_wikidata(label, kind="film"):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "format": "json",
        "limit": 5
    }
    headers = {
        "User-Agent": "MovieKG-StudentProject/1.0 (han.wang@edu.devinci.fr)"
    }
    r = requests.get(url, params=params, headers=headers, timeout=10)
    results = r.json().get("search", [])

    if not results:
        return None, 0.0

    # Prioritize descriptions containing the keyword "kind"
    for item in results:
        desc = item.get("description", "").lower()
        label_match = item.get("label", "").lower() == label.lower()
        if kind in desc:
            score = 1.0 if label_match else 0.8
            qid = item["id"]
            return f"http://www.wikidata.org/entity/{qid}", score

    # If no description with "film" is found, fallback to the first result
    best = results[0]
    score = 0.5
    return f"http://www.wikidata.org/entity/{best['id']}", score

# Test
uri, score = search_wikidata("Inception", kind="film")
print(uri, score)

http://www.wikidata.org/entity/Q25188 1.0


In [5]:
from rdflib import RDF, RDFS

def get_label(g, uri):
    label = g.value(uri, RDFS.label)
    return str(label) if label else None

def get_title(g, uri):
    title = g.value(uri, MVO.hasTitle)
    return str(title) if title else None

# Align movies
print("Aligning movies...")
for movie_uri in movies:
    title = get_title(g, movie_uri)
    if not title:
        continue
    
    wd_uri, score = search_wikidata(title, kind="film")
    
    if wd_uri and score >= 0.8:
        align_g.add((movie_uri, OWL.sameAs, URIRef(wd_uri)))
    
    mapping.append({
        "private_uri": str(movie_uri),
        "label": title,
        "type": "Movie",
        "wikidata_uri": wd_uri or "NOT_FOUND",
        "confidence": score
    })
    time.sleep(0.2)

print(f"Movies done. Aligned: {sum(1 for m in mapping if m['wikidata_uri'] != 'NOT_FOUND')}/{len(movies)}")

# Align directors (only directors)
print("Aligning directors...")
directors = set()
for movie_uri in movies:
    for director_uri in g.objects(movie_uri, MVO.hasDirector):
        directors.add(director_uri)

for person_uri in directors:
    name = get_label(g, person_uri)
    if not name:
        continue
    
    wd_uri, score = search_wikidata(name, kind="film director")
    
    if wd_uri and score >= 0.8:
        align_g.add((person_uri, OWL.sameAs, URIRef(wd_uri)))
    
    mapping.append({
        "private_uri": str(person_uri),
        "label": name,
        "type": "Director",
        "wikidata_uri": wd_uri or "NOT_FOUND",
        "confidence": score
    })
    time.sleep(0.2)

print(f"Directors done.")

# Save alignment file
align_g.serialize("alignment.ttl", format="turtle")

# Save mapping table as CSV
with open("mapping_table.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["private_uri","label","type","wikidata_uri","confidence"])
    writer.writeheader()
    writer.writerows(mapping)

print("Saved alignment.ttl and mapping_table.csv")

Aligning movies...
Movies done. Aligned: 152/163
Aligning directors...
Directors done.
Saved alignment.ttl and mapping_table.csv
